# Meme Sorting by Similarity
제품 설명(item_description)과 밈 정의(definition) + 예시(situation, dialogue_example)를 결합하여 유사도 비교 후 밈 정렬

In [1]:
print(1)

1


In [2]:
import os
import psycopg2
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# 한국어 임베딩 모델 로드
model = SentenceTransformer("dragonkue/BGE-m3-ko")
print(f"모델 로드 완료: dragonkue/BGE-m3-ko")

ModuleNotFoundError: No module named 'psycopg2'

In [26]:
# DB에서 memes + meme_examples 가져오기
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT", 5432),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

cur = conn.cursor()
cur.execute("""
    SELECT e.meme_id, e.example_id, e.situation, e.dialogue_example,
           m.meme_name, m.definition
    FROM meme_examples e
    JOIN memes m ON e.meme_id = m.meme_id
    ORDER BY e.meme_id, e.example_id
""")
rows = cur.fetchall()
cur.close()
conn.close()

print(f"총 {len(rows)}개 example 로드")
for r in rows[:3]:
    print(f"  meme_id={r[0]}, example_id={r[1]}, meme_name={r[4]}")
    print(f"    definition: {str(r[5])[:80] if r[5] else 'N/A'}")
    print(f"    situation: {r[2]}")
    print(f"    dialogue: {str(r[3])[:80] if r[3] else 'N/A'}")

총 303개 example 로드
  meme_id=1, example_id=1, meme_name=안성재옷 입히기
    definition: 안성재옷 입히기는 변형된 복장이나 의상을 입혀 안성재라는 가상 캐릭터를 꾸미는 과정을 통해 재미를 유발하는 인터넷 밈이다. 주로 소셜미디어에서 
    situation: 안성재에게 여러가지 옷을 입혀보며 코디하는 상황
    dialogue: 안성재에게 다양한 옷을 입혀보며 '코디해주기 시작함', '의외로 다 잘 어울리자' 등의 표현을 사용한다
  meme_id=1, example_id=2, meme_name=안성재옷 입히기
    definition: 안성재옷 입히기는 변형된 복장이나 의상을 입혀 안성재라는 가상 캐릭터를 꾸미는 과정을 통해 재미를 유발하는 인터넷 밈이다. 주로 소셜미디어에서 
    situation: 방송에서 안성재가 입고 나오는 옷에 대해 시청자들이 다음에는 무슨 옷을 입고 나올지 궁금해하는 상황
    dialogue: 영상에서 '다음 화엔 무슨 옷을 입고 나올까?'라는 자막이 나온다.
  meme_id=1, example_id=3, meme_name=안성재옷 입히기
    definition: 안성재옷 입히기는 변형된 복장이나 의상을 입혀 안성재라는 가상 캐릭터를 꾸미는 과정을 통해 재미를 유발하는 인터넷 밈이다. 주로 소셜미디어에서 
    situation: 안성재의 여러가지 슈트핏을 보여주는 영상
    dialogue: 안성재에게 여러 옷을 입혀보며 '안성재 옷 갈아입히기'를 보여준다.


In [27]:
def get_embeddings(texts):
    """로컬 한국어 임베딩 모델로 벡터를 생성한다."""
    return model.encode(texts, normalize_embeddings=True)

In [28]:
# 제품 설명 5개
item_descriptions = [
    "초경량 무선 블루투스 이어폰, 노이즈캔슬링 기능 탑재, 운동할 때 안 빠지는 이어폰",
    "유기농 수제 강아지 간식, 연어와 고구마로 만든 건강한 펫 트릿",
    "레트로 감성 필름 카메라, 감성 사진 찍기 좋은 빈티지 디자인",
    "1인용 캠핑 의자, 초경량 접이식, 배낭에 넣을 수 있는 사이즈",
    "대용량 보조배터리 20000mAh, 급속충전 지원, 여행 필수템",
]

# 각 example마다 definition + situation + dialogue_example을 결합
combined_texts = []
for r in rows:
    meme_id, example_id, situation, dialogue, meme_name, definition = r
    parts = []
    if definition:
        parts.append(f"밈 정의: {definition}")
    if situation:
        parts.append(f"상황: {situation}")
    if dialogue:
        parts.append(f"대화: {dialogue}")
    combined_texts.append(" | ".join(parts))

# situation만 따로 수집
situations = [r[2] for r in rows]

# 임베딩 생성 (아이템 5개 + 밈 텍스트는 한 번만)
item_embs = get_embeddings(item_descriptions)           # (5, dim)
combined_embs = get_embeddings(combined_texts)           # (N, dim)
situation_embs = get_embeddings(situations)              # (N, dim)

print(f"임베딩 완료: 제품 {len(item_descriptions)}개, combined {len(combined_texts)}개, situation {len(situations)}개")
print(f"임베딩 차원: {item_embs.shape[1]}")

임베딩 완료: 제품 5개, combined 303개, situation 303개
임베딩 차원: 1024


In [29]:
TOP_N = 10

for idx, item_desc in enumerate(item_descriptions):
    item_emb = item_embs[idx:idx+1]  # (1, dim)

    # --- combined (definition + situation + dialogue) ---
    sims_comb = cosine_similarity(item_emb, combined_embs)[0]

    best_comb = {}
    for i, row in enumerate(rows):
        meme_id, example_id, situation, dialogue, meme_name, definition = row
        if meme_id not in best_comb or sims_comb[i] > best_comb[meme_id]["similarity"]:
            best_comb[meme_id] = {
                "meme_name": meme_name, "situation": situation, "similarity": float(sims_comb[i])
            }
    sorted_comb = sorted(best_comb.values(), key=lambda x: x["similarity"], reverse=True)

    # --- situation only ---
    sims_sit = cosine_similarity(item_emb, situation_embs)[0]

    best_sit = {}
    for i, row in enumerate(rows):
        meme_id, example_id, situation, dialogue, meme_name, definition = row
        if meme_id not in best_sit or sims_sit[i] > best_sit[meme_id]["similarity"]:
            best_sit[meme_id] = {
                "meme_name": meme_name, "situation": situation, "similarity": float(sims_sit[i])
            }
    sorted_sit = sorted(best_sit.values(), key=lambda x: x["similarity"], reverse=True)

    # --- 출력 ---
    print(f"{'='*80}")
    print(f"제품 {idx+1}: {item_desc}")
    print(f"{'='*80}")
    print(f"\n  [combined: definition+situation+dialogue]")
    for rank, m in enumerate(sorted_comb[:TOP_N], 1):
        print(f"    {rank:>2}. [{m['similarity']:.4f}] {m['meme_name']:<25} | {m['situation'][:40]}")
    print(f"\n  [situation only]")
    for rank, m in enumerate(sorted_sit[:TOP_N], 1):
        print(f"    {rank:>2}. [{m['similarity']:.4f}] {m['meme_name']:<25} | {m['situation'][:40]}")
    print()

제품 1: 초경량 무선 블루투스 이어폰, 노이즈캔슬링 기능 탑재, 운동할 때 안 빠지는 이어폰

  [combined: definition+situation+dialogue]
     1. [0.3434] 골반통신                      | 패션 플랫폼 '무신사'에서 마케팅에 활용
     2. [0.3339] 골반이 안 멈추는데 어떡해            | 아이돌 그룹 엑스디너리 히어로즈의 신곡 홍보에 활용
     3. [0.3132] 내 골반이 멈추지 않는 탓일까? ㅜ.ㅜ     | 아이돌 마케팅 프로모션 문구에 활용
     4. [0.2832] 앙탈 챌린지                    | 애교부리는 상황
     5. [0.2699] 바라밤                       | 제주항공 승무원분들이 한 바라밤 챌린지를 보고 팬들이 좋아하는 모습
     6. [0.2600] 스피키 네르지 마세요               | 게임 '트릭컬 리바이브'의 마스코트가 전 세계적으로 인기를 끌며 밈으로 
     7. [0.2586] 요쇼하이하이                    | TikTok에서 친구들과 함께 춤을 추며 챌린지를 수행하는 영상에 활용
     8. [0.2548] 에브리바디 두 더 플랍              | 유튜브 영상에서 강아지들이 함께 놀며 플랍하는 모습을 보여주는 콘텐츠
     9. [0.2529] 햄부기햄북 햄북어                 | 여러 기업들이 이 밈을 브랜드 마케팅 전략에 통합하고 있습니다.
    10. [0.2504] 와쏘 베쏘                     | 틱톡 릴스를 휩쓴 '빠소 빼소 와쏘베이쏘' 노래에 대한 설명

  [situation only]
     1. [0.3519] 운동 많이 된다                  | 운동선수들이 운동하는 모습을 보여주는 영상
     2. [0.3138] 치킨 바나나                    | 제품 홍보 캠페인